<a href="https://colab.research.google.com/github/harini200614/Data-Visualization-lab/blob/main/DVT_Lab5_Google_Trends_Visualization_COLAB_231401033.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Import Libraries

In [ ]:
!pip -q install kagglehub

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

print("Libraries imported successfully.")

## 2. Download and Load the Real Kaggle Dataset

In [ ]:
dataset_path = kagglehub.dataset_download(
    "adrianjuliusaluoch/daily-google-search-trends-us"
)

csv_files = glob.glob(
    os.path.join(dataset_path, "**", "*.csv"),
    recursive=True
)

if not csv_files:
    raise FileNotFoundError("No CSV file was found in the Kaggle dataset.")

csv_path = csv_files[0]
df = pd.read_csv(csv_path)

print("Data loaded successfully!")
print("CSV file:", csv_path)
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

## 3. Prepare Columns for Visualization

The original Lab 5 expects:

- `location`
- `year`
- `category`
- `rank`
- `query`

The real Kaggle file may use different names. This cell detects the corresponding columns and creates any missing plotting fields from the actual data rather than inventing a replacement dataset.

In [ ]:
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

def find_column(candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None

query_col = find_column(["query", "search_query", "keyword", "term", "search_term"])
year_col = find_column(["year"])
date_col = find_column(["date", "date_recorded", "datetime", "timestamp", "time"])
rank_col = find_column(["rank", "ranking", "position"])
category_col = find_column(["category", "categories", "topic"])
location_col = find_column(["location", "country", "region", "geo"])

print("Detected columns:")
print("Query    :", query_col)
print("Year     :", year_col)
print("Date     :", date_col)
print("Rank     :", rank_col)
print("Category :", category_col)
print("Location :", location_col)

# Query
if query_col is None:
    raise ValueError("Could not identify a search-query column.")

df["query"] = df[query_col].astype(str).str.strip()

# Year
if year_col is not None:
    df["year"] = pd.to_numeric(df[year_col], errors="coerce")
elif date_col is not None:
    parsed_date = pd.to_datetime(df[date_col], errors="coerce")
    df["year"] = parsed_date.dt.year
else:
    raise ValueError("Could not identify a year/date column.")

# Rank
if rank_col is not None:
    df["rank"] = pd.to_numeric(df[rank_col], errors="coerce")
else:
    # If the real dataset has no rank field, rank search records within year.
    df["rank"] = df.groupby("year").cumcount() + 1
    print("No rank column was present; rank was generated from record order within year.")

# Category
if category_col is not None:
    df["category"] = df[category_col].astype(str).str.strip()
else:
    # Derive a transparent teaching category from query text.
    def assign_category(q):
        q = str(q).lower()
        groups = {
            "Technology": ["ai", "chatgpt", "python", "iphone", "android", "google", "microsoft", "tesla", "tech", "software"],
            "Sports": ["football", "soccer", "nba", "nfl", "cricket", "tennis", "baseball", "basketball", "match", "game"],
            "Entertainment": ["movie", "film", "music", "song", "concert", "netflix", "youtube", "celebrity", "tv", "show"],
            "News": ["news", "president", "election", "government", "war", "politics", "court", "law"],
        }
        for category, words in groups.items():
            if any(word in q for word in words):
                return category
        return "General"

    df["category"] = df["query"].apply(assign_category)
    print("No category column was present; category was derived transparently from search-query keywords.")

# Location
if location_col is not None:
    df["location"] = df[location_col].astype(str).str.strip()
else:
    df["location"] = "United States"
    print("No location column was present; the dataset geography is represented as United States.")

# Remove unusable rows for plotting
plot_df = df.dropna(subset=["year", "rank"]).copy()
plot_df["year"] = plot_df["year"].astype(int)

print("\nPrepared visualization columns:")
display(plot_df[["location", "year", "category", "rank", "query"]].head())

## 4. Line Chart – Trends over Years

In [ ]:
year_counts = plot_df["year"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.plot(
    year_counts.index,
    year_counts.values,
    marker="o"
)
plt.title("1. Line Chart: Google Trends Records over Years")
plt.xlabel("Year")
plt.ylabel("Number of Trend Records")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Bar Chart – Top Categories

In [ ]:
cat_counts = plot_df["category"].value_counts().head(5)

plt.figure(figsize=(9, 5))
plt.bar(
    cat_counts.index.astype(str),
    cat_counts.values
)
plt.title("2. Bar Chart: Top Categories")
plt.xlabel("Category")
plt.ylabel("Number of Records")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Pie Chart – Top Locations

In [ ]:
loc_counts = plot_df["location"].value_counts().head(5)

plt.figure(figsize=(7, 7))
plt.pie(
    loc_counts.values,
    labels=loc_counts.index.astype(str),
    autopct="%1.0f%%",
    startangle=90
)
plt.title("3. Pie Chart: Top Locations")
plt.tight_layout()
plt.show()

## 7. Histogram – Query Ranks Distribution

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(
    plot_df["rank"],
    bins=20,
    edgecolor="black"
)
plt.title("4. Histogram: Query Ranks Distribution")
plt.xlabel("Rank")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 8. Scatter Plot – Year vs Rank

In [ ]:
# Sample a maximum of 5,000 rows so the scatter plot remains readable
scatter_df = plot_df.sample(
    min(5000, len(plot_df)),
    random_state=42
)

plt.figure(figsize=(10, 5))
plt.scatter(
    scatter_df["year"],
    scatter_df["rank"],
    alpha=0.3
)
plt.title("5. Scatter Plot: Year vs Rank")
plt.xlabel("Year")
plt.ylabel("Rank")
plt.tight_layout()
plt.show()

## 9. Box Plot – Rank by Top Categories

In [ ]:
top_cats = plot_df["category"].value_counts().head(4).index.tolist()

data_by_cat = [
    plot_df.loc[plot_df["category"] == category, "rank"].dropna()
    for category in top_cats
]

plt.figure(figsize=(10, 6))
plt.boxplot(
    data_by_cat,
    labels=top_cats
)
plt.title("6. Box Plot: Rank by Top Categories")
plt.xlabel("Category")
plt.ylabel("Rank")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 10. Final Dataset Summary

In [ ]:
print("Final dataset used for visualization")
print("--------------------------------------")
print("Rows:", len(plot_df))
print("Columns:", len(plot_df.columns))
print("Year range:", plot_df["year"].min(), "to", plot_df["year"].max())

print("\nTop categories:")
print(plot_df["category"].value_counts().head(5))

print("\nTop locations:")
print(plot_df["location"].value_counts().head(5))